# 04 — Genie Ontology and Genie One

**Time: about 13 minutes.**
Covers: Genie Ontology (Public Preview) · Unity Catalog semantics vs. inferred context · Certification · Citations · Genie One (Generally Available) · Acting through Genie One · Governance

### What you will be able to do afterwards

- Inspect which sources Genie relied on for an answer, and curate the signals that rank them.
- Use Genie One, the business-facing entry point, and have it change a Databricks object on your behalf.
- State which Unity Catalog guarantees carry through to every answer.

## The concepts

**Genie Ontology** is the context layer behind the Genie family. It combines two kinds of context:

| Kind | Examples | Who controls it |
|---|---|---|
| **Unity Catalog semantics** (modeled) | Metric views, domains, Pages | You define, govern and certify it |
| **Inferred context** (snippets) | Definitions and rules Genie extracts from metric views, dashboards, SQL queries and Genie Agents | Genie maintains it from your assets and usage |

Each snippet gets an **authority score** from where it came from, how much it is used, and how fresh it
is. Certified assets rank higher; deprecated ones are steered away from. Snippets are **gated by Unity
Catalog permissions**: Genie only uses context from assets the asking user can see. Answers show
**citation** icons listing the sources used.

There is no editor for individual snippets. You curate the ontology by changing what it reads:
certification, descriptions, metric views and Pages.

**Genie One** (generally available; it was previously called **Databricks One**) is the simplified
UI for business users at `<workspace-url>/one`. Its chat searches your Genie Agents first, then
dashboards, queries and metric views, answering through the ontology. It can also **act**: edit Genie
Agents, schedule recurring tasks, draft documents, and use external tools (Slack, GitHub, Google
Workspace, Microsoft 365, custom MCP services) through MCP connectors governed by Unity Gateway.

### Availability — read before you start

| Feature | Status | Needs |
|---|---|---|
| Genie One chat | GA | Consumer access entitlement or higher; CAN USE on a SQL warehouse; partner-powered AI features on |
| Genie Ontology, citations | Public Preview | Same as above; may need enabling by an admin |
| Certification tag | GA | **ASSIGN** on the `system.certification_status` governed tag (admin grant) |
| Pages | Beta | Enabled on the Previews page, plus a domain to put them in |

If something is not available in your workspace, every step below says what to do instead: record
`availability="unavailable"` with a note on what you saw. The checks accept that, and your instructor
can tell a disabled feature from a skipped step.

In [ ]:
from helpers import utils, ontology, observations, genie_agents, ground_truth

cfg = utils.use_lab_schema()
spark = utils.spark
print(f"Working in {cfg['catalog']}.{cfg['schema']}")

In [ ]:
print(f"Genie One: {ontology.genie_one_url()}")
agent_id = genie_agents.registered_id("metric_view_agent")
AGENT_TITLE = genie_agents.get_space(agent_id)["title"] if agent_id else None
print(f"Your Exercise 1 agent: {AGENT_TITLE or 'NOT REGISTERED - finish 02 first'}")

## Step 1 — The modeled context Genie starts from

Two objects compete to answer "revenue": your metric view, and the legacy summary table with a column
literally called `revenue`. Compare what each tells a reader.

In [ ]:
for name in (utils.METRIC_VIEW, ontology.LEGACY_VIEW):
    print(f"{name}\n  comment:       {ontology.comment_of(name)}\n  certification: {ontology.certification(name)}\n")

## Step 2 — Inspect what the ontology has inferred

Your Exercise 1–3 work (queries you ran, the metric view, both agents) is now input to the ontology.

1. Open Genie One (URL above; or the app switcher in the upper-right corner > **Genie One**).
2. Click **New chat**. Do not pick an agent.
3. Ask: **How does Lumen Outfitters define revenue?**
4. Click the **citation icons** in the answer and list every source it cites: the metric view? your
   metric-view agent? the raw agent? `lab_daily_sales_summary`? past queries?

Inferred snippets are built asynchronously. If the answer has no citations, or Genie One is not
available, record that instead.

**TO DO**

In [ ]:
observations.record("04", "inspect_ontology", question_key="revenue_definition",
    sources_cited="",           # TO DO: every source the citations listed
    availability="available",   # or "unavailable" if Genie One / citations are not enabled
    notes="")                   # TO DO: what definition did it give? did the legacy table appear?

## Step 3 — Curate

**3A — Certify the governed asset, deprecate the bait.** This is the strongest signal. It needs ASSIGN
on the `system.certification_status` governed tag; without it the cell prints what to ask an admin for
and you continue with cell 3B next.

In [ ]:
ontology.set_certification(utils.METRIC_VIEW, "certified")
ontology.set_certification(ontology.LEGACY_VIEW, "deprecated")

If the 3A cell fails due to permission error, skip it and run cell 3B. The notebook is designed for this fallback. 3B does description-based curation and needs no extra grant. Genie will use the descriptions instead of the certified/deprecated tags. The tag is described as the strongest signal, so results may be slightly weaker without it.

**3B — Fix the description (no extra grant needed).** The legacy view's comment is what a reader, human
or Genie, sees first. Make it say what the table really is and where to go instead.

In [ ]:
ontology.annotate_legacy_summary()

**3C — Optional: certify your agent and write a Page.**

- *Certify the agent:* open your Exercise 1 agent, kebab menu > **Assign certification** > **Certified**
  (same ASSIGN requirement as 3A).
- *Page (Beta):* **Discover > Pages > Create page**. Pick the training domain your instructor named.
  Title it `Net revenue - <your name>`, add synonyms *revenue, net sales*, a one-paragraph definition,
  and the metric view and your agent under **Related assets**. Click **Save as draft**: a draft Page is
  used only in your own Genie One chats, so you will not change anyone else's answers.

If Pages or the domain are not there, skip this; 3A or 3B is enough for the checks.

In [ ]:
print(ontology.curation_status())

## Step 4 — Make Genie One act

Genie One can change Databricks objects for you, inside your permissions. Ask it to add a rule to your
Exercise 1 agent. Paste the line printed below into a Genie One chat. It shows the change and asks for
confirmation; approve it.

In [ ]:
print(ontology.genie_one_instruction(AGENT_TITLE or '<your Exercise 1 agent title>'))

In [ ]:
genie_agents.describe(genie_agents.registered_id("metric_view_agent"))

The marker `(added by Genie One)` should now appear in the agent's instructions; the checks look for it.
If Genie One cannot edit agents in your workspace, record what you saw instead:

In [ ]:
# observations.record("04", "genie_one_action", availability="unavailable",
#                     notes="TO DO: what happened when you asked")

*Optional:* in Genie One, click **+** in the message box and connect an external tool if your workspace
offers one (Slack, GitHub, ...). Those actions go through MCP connectors: each user signs in
individually, and Unity Gateway authorizes every tool call and logs it.

## Step 5 — What governance carries over

| Guarantee | What it means for an answer |
|---|---|
| Data access is the **end user's** UC identity | Genie queries run with the asker's permissions; row filters and column masks apply. A Genie Agent embeds only the author's *warehouse* credentials, never data access. |
| Ontology snippets are **permission-gated** | Context extracted from an asset is only used for users who can see that asset. |
| Certification is a **governed tag** | Only principals with ASSIGN can certify, so "certified" means something. |
| Actions use **your** permissions | Editing an agent needs CAN EDIT on it; external tools need your own sign-in and are audited. |
| Everything is **attributed** | Genie queries appear in query history under the asking user. |

---

## Challenge 4

Get Genie One to give the **governed** answer to the revenue question.

1. In a **new** Genie One chat (no agent picked), ask: *What was our revenue in the last 90 days?*
2. Check the number and the citations. If it is not net revenue, strengthen your curation (Step 3),
   wait a few minutes for re-indexing, and ask again in a new chat.
3. Record the number and the sources it cited.

**TO DO**

In [ ]:
observations.record("04", "genie_one_answer", question_key="revenue_90d",
    answer_value=None,          # TO DO
    sources_cited="",           # TO DO
    availability="available",   # or "unavailable", with notes
    notes="")                   # TO DO: did it route to your agent? how many attempts?

## Checks

In [ ]:
from helpers import test_runner
test_runner.run("04-genie-ontology-and-genie-one")